# Install and prepare libraries

The "names" package in Python is install to generate syntetic names for patients

In [1]:
pip install names

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.1/789.1 kB 4.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for names: filename=names-0.3.0-py3-none-any.whl size=803749 sha256=0a6160ce26eb6ebe52ae465fd8c6cfcdcdee68828b46f2459e5e73fd61be4454
  Stored in directory: /root/.cache/pip/wheels/bd/d4/8f/03181acc8ebae4403c4a16423e9f68a0b0e65496d607d43462
Successfully built names


Furrther the Python libraries such as pandas, numpy, names, random, and datetime are imported

In [2]:
import pandas as pd
import numpy as np
import random  # Import the random module
import names   # Assuming you have the 'names' module installed
import uuid


We donwload a medical archive CSV file from a URL, with specific delimiter, and displays the first few rows of the resulting DataFrame containing medical diagnosis and symptom data.

In [3]:
Medical_archieve=pd.read_csv("https://pvexpect.com/Healthcare/Medical_archieve.csv", delimiter=";")
Medical_archieve.head(1)

,Diagnosis,Symptom,Examination,Examination_2,Behandlding,Avdeling,Common/uncommon,"Sannsynlighet for kritisk utfall, når symptom er påvist",Kritiskhet kategori,Interview reveal,...,Diagnoses,Diagnoses_temp,Required Room(s),Departement(s),Required Equipment(s),Legehåndbok,Virtuell sykehus,Kostnader($),Sykdoms beskrivelse (Engelsk),Sykdoms beskrivelse (Norsk)
0,Acute Rheumatic Fever,Rheumatic Fever,an echocardiogram,NaN,antibiotics,Cardiology,Uncommon,100 %,Medium Hazard,False,...,"Bacterial tonsillitis, Laryngitis, Bronchitis,...",0,"Any office, Diagnostic unit",Any,0.0,https://nhi.no/sykdommer/hjertekar/ulike-sykdo...,0,1100.0,Acute rheumatic fever is an inflammatory disea...,Akutt revmatisk feber er en inflammatorisk syk...


In [4]:
# Assuming you have already loaded the dataset into Medical_archieve
column_names = Medical_archieve.columns

# Print the column names
print(column_names)

Index(['Diagnosis', 'Symptom', 'Examination', 'Examination_2', 'Behandlding',
       'Avdeling', 'Common/uncommon',
       'Sannsynlighet for kritisk utfall, når symptom er påvist',
       'Kritiskhet kategori', 'Interview reveal', 'Symptom.1', 'ICON',
       'IsMainSymptom', 'DiscomfortLevel', 'PatientComplains', 'Hazard',
       'PatientMobility', 'Examination 1', 'Examination 2', 'Examination 3',
       'Treatment', 'ShameLevel', 'WalkAnim', 'WalkSpeedModifier',
       'CollapseSymptom', 'RiskOfCollapseStartHours', 'RiskOfCollapseEndHours',
       'CannotTalk', 'RiskOfDeathStartHours', 'RiskOfDeathEndHours',
       'CollapseProcedure', 'PlanAllTreatments', 'BleedingLevel',
       'Behandling (Engelsk)', 'Behandling (Norsk)',
       'Behandling beskrivelse (Engelsk)', 'Behandling beskrivelse (Norsk)',
       'Krav', 'Surgery room', 'Diagnoses', 'Diagnoses_temp',
       'Required Room(s)', 'Departement(s)', 'Required Equipment(s)',
       'Legehåndbok', 'Virtuell sykehus', 'Kostnader(

In [5]:
# Assuming you have already loaded the dataset into Medical_archieve
num_columns = Medical_archieve.shape[1]

# Print the number of columns
print("Number of columns:", num_columns)

Number of columns: 49


Then we develop a function, `generate_patient()`, that creates a random patient profile by selecting a disease, generating various patient attributes such as name, planned appointment, hazard level, timeslot, arrival time, revealed and unrevealed symptoms, and diagnosis based on a medical archive dataset.

Her har vi valgt features som er relevante til hospital-sim i notebook 2

In [6]:
def generate_patient():
    # Bruk symptomer som kan avdekkes i intervjuet
    tilgjengelige = Medical_archieve[
        (Medical_archieve["Interview reveal"] == True)
        & Medical_archieve["Diagnosis"].notna()
        & Medical_archieve["Symptom"].notna()
    ].copy()

    # Fjern tomme symptomer og duplikater
    tilgjengelige["Symptom"] = (
        tilgjengelige["Symptom"].astype(str).str.strip()
    )
    tilgjengelige = tilgjengelige[
        tilgjengelige["Symptom"] != ""
    ].drop_duplicates(subset=["Diagnosis", "Symptom"])

    # Bruk bare diagnoser med minst fire ulike symptomer
    antall_symptomer = tilgjengelige.groupby("Diagnosis").size()
    aktuelle_diagnoser = antall_symptomer[
        antall_symptomer >= 4
    ].index.tolist()

    if not aktuelle_diagnoser:
        raise ValueError(
            "Ingen diagnoser har fire tilgjengelige intervjusymptomer."
        )

    diagnose = random.choice(aktuelle_diagnoser)

    symptomrader = tilgjengelige[
        tilgjengelige["Diagnosis"] == diagnose
    ]

    # Velg fire symptomer fra samme diagnose
    valgte = symptomrader.sample(n=4)
    symptomer = valgte["Symptom"].tolist()

    # Bruk et hovedsymptom som hovedplage hvis det finnes
    hovedsymptomer = valgte[
        valgte["IsMainSymptom"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("TRUE")
    ]

    hovedplage = (
        str(hovedsymptomer.iloc[0]["Symptom"]).strip()
        if not hovedsymptomer.empty
        else symptomer[0]
    )

    # Legg hovedplagen først, slik at den beholdes i versjonen med to symptomer
    symptomer = [hovedplage] + [
        symptom for symptom in symptomer
        if symptom != hovedplage
    ]

    # Syntetisk alder og kjønn til simuleringen
    kjonn = random.choice(["Male", "Female"])
    alder = random.randint(18, 90)

    # Standardverdier for vitale tegn
    puls = 80  # bpm
    blodtrykk = {"systolic": 120, "diastolic": 80}
    temperatur = 98.6  # Fahrenheit
    pustefrekvens = 15  # pust per minutt

    # Gjør symptomnavn uavhengige av store/små bokstaver
    symptomnavn = {
        symptom.strip().lower() for symptom in symptomer
    }

    if "high fever" in symptomnavn:
        temperatur = 103.10
    elif "fever" in symptomnavn:
        temperatur = 101.3

    if "tachycardia" in symptomnavn:
        puls = 110

    if "bradycardia" in symptomnavn:
        puls = 40

    if "hypertension" in symptomnavn:
        blodtrykk = {"systolic": 140, "diastolic": 90}

    if "hypotension" in symptomnavn:
        blodtrykk = {"systolic": 90, "diastolic": 60}

    if diagnose.strip().lower() == "stroke":
        blodtrykk = {"systolic": 160, "diastolic": 90}

    if diagnose.strip().lower() in [
        "mild frostbite on hands",
        "mild frostbite on legs",
    ]:
        puls = 100
        temperatur = 95

    if diagnose.strip().lower() in [
        "severe frostbite on hands",
        "severe frostbite on legs",
    ]:
        puls = 40
        temperatur = 89.6

    if "bradypnea" in symptomnavn:
        pustefrekvens = 8

    if "tachypnea" in symptomnavn:
        pustefrekvens = 24

    pasient_id = str(uuid.uuid4())

    # Pasientopplysninger til simuleringen – uten fasitdiagnose
    profil = {
        "profile_id": pasient_id,
        "name": names.get_full_name(gender=kjonn.lower()),
        "age": alder,
        "gender": kjonn,
        "chief_complaint": hovedplage,
        "symptoms": symptomer,
        "medical_history": [],
        "current_medications": [],
        "allergies": [],
        "vital_signs": {
            "heart_rate": puls,
            "blood_pressure": blodtrykk,
            "temperature": temperatur,
            "Respiratory rate": pustefrekvens,
        },
    }

    # Separat fasit til kontroll etter simuleringen
    fasit = {
        "profile_id": pasient_id,
        "name": profil["name"],
        "reference_diagnosis": diagnose,
    }

    return profil, fasit

Lager en tabell med pasienter. Antallet kan justeres.

In [7]:
from copy import deepcopy

antall_pasienter = 20

pasienter_2 = []
pasienter_4 = []
fasitliste = []

for _ in range(antall_pasienter):
    profil, fasit = generate_patient()

    # Legg hovedplagen først, slik at den er med i begge versjoner
    hovedplage = profil["chief_complaint"]
    profil["symptoms"] = [hovedplage] + [
        symptom
        for symptom in profil["symptoms"]
        if symptom != hovedplage
    ]

    # Samme pasient, men med bare de to første symptomene
    profil_2 = deepcopy(profil)
    profil_2["symptoms"] = profil["symptoms"][:2]

    pasienter_2.append(profil_2)
    pasienter_4.append(profil)
    fasitliste.append(fasit)

display(pd.DataFrame(pasienter_2))
display(pd.DataFrame(pasienter_4))
display(pd.DataFrame(fasitliste))

,profile_id,name,age,gender,chief_complaint,symptoms,medical_history,current_medications,allergies,vital_signs
0,fcf1197f-4c25-4a64-a91b-f4f18408eadc,John Miller,33,Male,Fever,"[Fever, Nausea]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
1,0e408cc6-a6b4-429a-aeeb-88f1888da86e,Julio Rice,18,Male,Swelling,"[Swelling, Desquamation]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
2,17530cf9-a12b-4832-b6bc-1549f5f46a78,Keith Myers,36,Male,Redness,"[Redness, Breathing Problems]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
3,7021e0af-1ffa-4f05-aea3-115ca3e0eeaf,Roger Reveles,52,Male,Tenderness,"[Tenderness, Spasms]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
4,f5169136-961a-4b51-833a-f2249becc835,Debra Givens,49,Female,Red Eye,"[Red Eye, Fatigue]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
5,f32f0216-e236-4444-9ea4-163602050637,Hung Rhyner,88,Male,Elbow Pain,"[Elbow Pain, Hand Weakness]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
6,a9267ecd-6357-40cb-8ada-d7e1e83dd2d8,Lula Carter,52,Female,Headache,"[Headache, Fever]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
7,33745826-4326-46cc-8891-b9066c301e17,Rebecca Fanning,52,Female,Swollen Lymph Nodes,"[Swollen Lymph Nodes, Fainting]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
8,9c1c0d28-2b61-4c45-a420-d5221a65d578,Norma Jones,69,Female,Dizziness,"[Dizziness, Hemoptysis]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
9,acfba1ed-fefa-4f0c-ad2c-f76fcbe39fd8,Michael Berka,76,Male,Eschar,"[Eschar, Muscle Pain]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."


,profile_id,name,age,gender,chief_complaint,symptoms,medical_history,current_medications,allergies,vital_signs
0,fcf1197f-4c25-4a64-a91b-f4f18408eadc,John Miller,33,Male,Fever,"[Fever, Nausea, Fatigue, Headache]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
1,0e408cc6-a6b4-429a-aeeb-88f1888da86e,Julio Rice,18,Male,Swelling,"[Swelling, Desquamation, Redness, Severe Arm P...",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
2,17530cf9-a12b-4832-b6bc-1549f5f46a78,Keith Myers,36,Male,Redness,"[Redness, Breathing Problems, Bruise, Tenderness]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
3,7021e0af-1ffa-4f05-aea3-115ca3e0eeaf,Roger Reveles,52,Male,Tenderness,"[Tenderness, Spasms, Foot Injury, Warmth]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
4,f5169136-961a-4b51-833a-f2249becc835,Debra Givens,49,Female,Red Eye,"[Red Eye, Fatigue, Irritable Cough, Sneezing]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
5,f32f0216-e236-4444-9ea4-163602050637,Hung Rhyner,88,Male,Elbow Pain,"[Elbow Pain, Hand Weakness, Ulnar Claw, Muscle...",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
6,a9267ecd-6357-40cb-8ada-d7e1e83dd2d8,Lula Carter,52,Female,Headache,"[Headache, Fever, Dry Cough, Nasal Congestion]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
7,33745826-4326-46cc-8891-b9066c301e17,Rebecca Fanning,52,Female,Swollen Lymph Nodes,"[Swollen Lymph Nodes, Fainting, Fever, Chest P...",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
8,9c1c0d28-2b61-4c45-a420-d5221a65d578,Norma Jones,69,Female,Dizziness,"[Dizziness, Hemoptysis, Chest Pain, Breathing ...",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
9,acfba1ed-fefa-4f0c-ad2c-f76fcbe39fd8,Michael Berka,76,Male,Eschar,"[Eschar, Muscle Pain, Petechiae, Fever]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."


,profile_id,name,reference_diagnosis
0,fcf1197f-4c25-4a64-a91b-f4f18408eadc,John Miller,West Nile Fever
1,0e408cc6-a6b4-429a-aeeb-88f1888da86e,Julio Rice,Severe thermal Burn
2,17530cf9-a12b-4832-b6bc-1549f5f46a78,Keith Myers,Broken Ribs
3,7021e0af-1ffa-4f05-aea3-115ca3e0eeaf,Roger Reveles,Metatarsal Bones Fracture
4,f5169136-961a-4b51-833a-f2249becc835,Debra Givens,Hay Fever
5,f32f0216-e236-4444-9ea4-163602050637,Hung Rhyner,Cubital Tunnel Syndrome
6,a9267ecd-6357-40cb-8ada-d7e1e83dd2d8,Lula Carter,Common Cold
7,33745826-4326-46cc-8891-b9066c301e17,Rebecca Fanning,Sarcoidosis - Heart
8,9c1c0d28-2b61-4c45-a420-d5221a65d578,Norma Jones,Pulmonary Hypertension
9,acfba1ed-fefa-4f0c-ad2c-f76fcbe39fd8,Michael Berka,Japanese Spotted Fever


Laster ned pasientene som en fil, som må legges inn i notebook 2

In [8]:
import json
from google.colab import files

json_filer = {
    "pasienter_2_symptomer.json": pasienter_2,
    "pasienter_4_symptomer.json": pasienter_4,
    "diagnosefasit.json": fasitliste,
}

for filnavn, innhold in json_filer.items():
    with open(filnavn, "w", encoding="utf-8") as fil:
        json.dump(
            innhold,
            fil,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )

    files.download(filnavn)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
files.download("pasienter_2_symptomer.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
files.download("pasienter_4_symptomer.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>